In [ ]:
!python -m pip install requests

In [ ]:

import csv
import json
from datetime import datetime, timezone
from pathlib import Path

import requests

BASE_URL = "https://api.gbif.org/v1"
OUTPUT_DIR = Path("outputs/gbif_pilot")
LIMIT = 20
MAX_SPECIES = 10
TIMEOUT = 30

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_json(url, params=None):
    response = requests.get(
        url,
        params=params,
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def save_csv(filename, rows):
    if not rows:
        print(f"Sin datos para {filename}")
        return

    fields = list(dict.fromkeys(
        key for row in rows for key in row.keys()
    ))

    with (OUTPUT_DIR / filename).open(
        "w", newline="", encoding="utf-8-sig"
    ) as file:
        writer = csv.DictWriter(
            file, fieldnames=fields, extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(rows)


def main():
    started_at = datetime.now(timezone.utc).isoformat()

    # 1. Consultar registros de aves en Colombia.
    occurrence_response = get_json(
        f"{BASE_URL}/occurrence/search",
        params={
            "country": "CO",
            "taxonKey": 212,
            "limit": LIMIT,
            "offset": 0,
        },
    )

    raw_occurrences = occurrence_response.get("results", [])

    # Guardar la respuesta original para poder auditarla.
    (OUTPUT_DIR / "occurrences_raw.json").write_text(
        json.dumps(
            occurrence_response,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    occurrence_rows = []
    species_keys = []

    for item in raw_occurrences:
        species_key = item.get("speciesKey")

        if species_key is not None and species_key not in species_keys:
            species_keys.append(species_key)

        occurrence_rows.append({
            "gbif_id": item.get("key"),
            "scientific_name": item.get("scientificName"),
            "species": item.get("species"),
            "event_date": item.get("eventDate"),
            "country": item.get("country"),
            "state_province": item.get("stateProvince"),
            "locality": item.get("locality"),
            "latitude": item.get("decimalLatitude"),
            "longitude": item.get("decimalLongitude"),
            "dataset_key": item.get("datasetKey"),
            "publisher": item.get("publishingOrg"),
            "source_url": (
                f"https://www.gbif.org/occurrence/{item['key']}"
                if item.get("key") is not None else None
            ),
        })

    save_csv("occurrences.csv", occurrence_rows)

    # 2. Consultar datos taxonómicos de especies identificadas.
    species_rows = []
    errors = []

    for species_key in species_keys[:MAX_SPECIES]:
        try:
            details = get_json(
                f"{BASE_URL}/species/{species_key}"
            )

            vernacular_response = get_json(
                f"{BASE_URL}/species/{species_key}/vernacularNames"
            )

            names = vernacular_response.get("results", [])
            common_names = [
                name.get("vernacularName")
                for name in names
                if name.get("vernacularName")
            ]

            species_rows.append({
                "gbif_species_key": species_key,
                "scientific_name": details.get("scientificName"),
                "canonical_name": details.get("canonicalName"),
                "taxonomic_rank": details.get("rank"),
                "taxonomic_status": details.get("taxonomicStatus"),
                "kingdom": details.get("kingdom"),
                "family": details.get("family"),
                "common_names": " | ".join(common_names),
                "gbif_taxon_url": (
                    f"https://www.gbif.org/species/{species_key}"
                ),
            })

        except requests.RequestException as error:
            errors.append({
                "species_key": species_key,
                "error": str(error),
            })

    save_csv("species.csv", species_rows)

    summary = {
        "started_at_utc": started_at,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": "GBIF API",
        "country_filter": "CO",
        "taxon_filter": "Aves (taxonKey=212)",
        "requested_limit": LIMIT,
        "occurrences_returned": len(occurrence_rows),
        "unique_species_found": len(species_keys),
        "species_enriched": len(species_rows),
        "species_enrichment_limit": MAX_SPECIES,
        "errors": errors,
        "status": "completed_with_errors" if errors else "completed",
        "note": (
            "Muestra exploratoria; no representa el total de "
            "registros de aves de Colombia."
        ),
    }

    (OUTPUT_DIR / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"\nArchivos generados en: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()


{
  "started_at_utc": "2026-10-16T22:25:44.115815+00:00",
  "finished_at_utc": "2026-10-16T22:25:55.309951+00:00",
  "source": "GBIF API",
  "country_filter": "CO",
  "taxon_filter": "Aves (taxonKey=212)",
  "requested_limit": 20,
  "occurrences_returned": 20,
  "unique_species_found": 17,
  "species_enriched": 10,
  "species_enrichment_limit": 10,
  "errors": [],
  "status": "completed",
  "note": "Muestra exploratoria; no representa el total de registros de aves de Colombia."
}

Archivos generados en: /content/outputs/gbif_pilot
